![LogoUC3M](https://upload.wikimedia.org/wikipedia/commons/thumb/a/a6/Acr%C3%B3nimo_y_nombre_de_la_UC3M.svg/320px-Acr%C3%B3nimo_y_nombre_de_la_UC3M.svg.png)

# PRIMERA PRACTICA - ENTRENAMIENTO Y ANALISIS DEL MODELO

### **Grupo 82 - Equipo 15**

*   Ariana Cornejo Infante,     100522121, 100522121@alumnos.uc3m.es
*   Francisco Pérez Sokolowski, 100522254, 100522254@alumnos.uc3m.es

In [9]:
# Imports necesarios
import pandas as pd
import numpy as np
import requests
import io
from joblib import load

Se realizará la carga del modelo final entrenado en el Notebook 1 (`modelo_final.joblib`).

In [10]:
# --- CARGAR EL MODELO FINAL ---
# Ruta del modelo exportado desde el Notebook 1
import requests

MODEL_PATH = "modelo_final.joblib"
url_model = "https://github.com/100522254/Proyecto-Aprendizaje-Autom-tico/raw/main/modelo_final.joblib"
url_comp = "https://github.com/100522254/Proyecto-Aprendizaje-Autom-tico/raw/main/bank_competition.pkl"

# Descargar y cargar modelo
response = requests.get(url_model)
with open(MODEL_PATH, "wb") as f:
    f.write(response.content)

print("---- CARGA DEL MODELO FINAL ----\n")
print("Modelo descargado correctamente.")

try:
    pack = load(MODEL_PATH)
    final_pipeline  = pack["pipeline"]
    feature_metadata = pack["feature_metadata"]
    classes_         = pack["classes_"]
    print(f"Modelo cargado correctamente desde '{MODEL_PATH}'.\n")
    print(f"Clases del modelo: {classes_}\n")
except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró '{MODEL_PATH}'. "
        "Asegúrate de ejecutar primero el Notebook 1 para generar el modelo."
    )

# --- CARGA DE DATOS DE COMPETICIÓN ---

def load_pkl_from_url(url):
    """Carga un archivo .pkl desde una URL y lo devuelve como DataFrame."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        return pd.read_pickle(io.BytesIO(response.content))
    except Exception as e:
        print(f"Error cargando {url}: {e}")
        return None

df_comp = load_pkl_from_url(url_comp)

if df_comp is None:
    raise RuntimeError("No se pudo cargar el dataset de competición. Verifica el enlace.")

print("---- CARGA DE DATOS DE COMPETICIÓN ----\n")
print(f"Dataset de competición cargado: {df_comp.shape[0]} instancias, {df_comp.shape[1]} variables.")
print(df_comp.head(3).to_string())

# --- Preproceso previo de la variable 'pdays_contacted' ---
# Replicar el mismo preproceso aplicado en el Notebook 1
df_comp['pdays_contacted'] = np.where(df_comp['pdays'] == -1, 0, 1)
print("Columna 'pdays_contacted' añadida al dataset de competición.")

---- CARGA DEL MODELO FINAL ----

Modelo descargado correctamente.
Modelo cargado correctamente desde 'modelo_final.joblib'.

Clases del modelo: ['no', 'yes']

---- CARGA DE DATOS DE COMPETICIÓN ----

Dataset de competición cargado: 162 instancias, 16 variables.
      age         job  marital  education default  balance housing loan   contact  day month  duration  campaign  pdays  previous poutcome
5553   43  management  married   tertiary      no       78     yes   no  cellular   21   nov        36         1    109         1    other
915    34   housemaid  married  secondary      no        0     yes   no   unknown   30   oct       154         1     -1         0  unknown
7652   54  technician  married  secondary      no     3323     yes  yes  cellular    8   apr        59         3     -1         0  unknown
Columna 'pdays_contacted' añadida al dataset de competición.


### 6.3. Predicciones para la competición
Se utilizará el modelo final para obtener predicciones para el conjunto de datos de la competición, y se guardarán en 'predicciones.csv'.

In [11]:
predicciones_num = final_pipeline.predict(df_comp)

# Convertir 0/1 → 'no'/'yes'
pred_labels = np.where(predicciones_num == 1, "yes", "no")
pred_df = pd.DataFrame({"deposit": pred_labels})

print("---- PREDICCIONES PARA LA COMPETICIÓN ----\n")
print(f"Total de predicciones generadas: {len(pred_df)}")
print("\nDistribución de predicciones:")
print(pred_df["deposit"].value_counts().to_string())
print(f"\nPorcentaje YES: {(pred_df['deposit'] == 'yes').mean() * 100:.2f}%")
print(f"Porcentaje NO : {(pred_df['deposit'] == 'no').mean() * 100:.2f}%")

# Generar el .csv para guardar las predicciones
OUTPUT_CSV = "predicciones.csv"
pred_df.to_csv(OUTPUT_CSV, index=False)

print(f"Archivo '{OUTPUT_CSV}' guardado correctamente.")

---- PREDICCIONES PARA LA COMPETICIÓN ----

Total de predicciones generadas: 162

Distribución de predicciones:
deposit
no    162

Porcentaje YES: 0.00%
Porcentaje NO : 100.00%
Archivo 'predicciones.csv' guardado correctamente.


### 6.4. Verificación de predicciones manuales vs modelo
Se mostrará cómo usar el modelo para predecir sobre instancias nuevas, y permite verificar que las predicciones
del modelo y de Streamlit coincidan.

In [12]:
# Tomar dos instancias del dataset de competición como ejemplo
ej1 = df_comp.iloc[[0]].copy()
ej2 = df_comp.iloc[[5]].copy()

# Usar la pipeline y asignar la etiqueta
pred1 = final_pipeline.predict(ej1)
pred2 = final_pipeline.predict(ej2)
label1 = "yes" if pred1[0] == 1 else "no"
label2 = "yes" if pred2[0] == 1 else "no"

print("---- VERIFICACIÓN DE PREDICCIONES ----\n")
print(f"Instancia 1 (índice 0):")
print(ej1.to_string())
print(f"→ Predicción del modelo: '{label1}'")

print(f"\nInstancia 2 (índice 5):")
print(ej2.to_string())
print(f"→ Predicción del modelo: '{label2}'")

print("\n[Estas predicciones deben coincidir con las que muestra la app Streamlit para los mismos datos.]")

---- VERIFICACIÓN DE PREDICCIONES ----

Instancia 1 (índice 0):
      age         job  marital education default  balance housing loan   contact  day month  duration  campaign  pdays  previous poutcome  pdays_contacted
5553   43  management  married  tertiary      no       78     yes   no  cellular   21   nov        36         1    109         1    other                1
→ Predicción del modelo: 'no'

Instancia 2 (índice 5):
      age         job  marital education default  balance housing loan   contact  day month  duration  campaign  pdays  previous poutcome  pdays_contacted
7779   35  management  married  tertiary      no      178      no   no  cellular   14   aug        76         2     -1         0  unknown                0
→ Predicción del modelo: 'no'

[Estas predicciones deben coincidir con las que muestra la app Streamlit para los mismos datos.]
